# FRED API Demo / FRED 数据源示例

这个 notebook 用 FRED 的几个宏观序列做一个最小演示，快速看数据覆盖区间、最新值和长期趋势。  
This notebook gives a minimal demo of a few FRED macro series so you can quickly inspect coverage, recent values, and long-run trends.

示例序列包括：  
Sample series include:

- `GDP`
- `UNRATE`
- `FEDFUNDS`


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("ggplot")

SERIES = {
    "GDP": "Real Gross Domestic Product / 实际 GDP",
    "UNRATE": "Unemployment Rate / 失业率",
    "FEDFUNDS": "Effective Federal Funds Rate / 联邦基金利率",
}


def load_fred_series(series_id: str) -> pd.DataFrame:
    # FRED 可以直接通过 CSV 拉取，这样 notebook 不需要额外依赖。
    # FRED can be pulled as a simple CSV, which keeps the notebook dependency-light.
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    frame = pd.read_csv(url)

    date_col = next((col for col in ["DATE", "date", "observation_date"] if col in frame.columns), None)
    if date_col is None:
        raise ValueError(f"Could not find a date column in FRED response for {series_id}. Columns: {list(frame.columns)}")

    value_col = next((col for col in frame.columns if col != date_col), None)
    if value_col is None:
        raise ValueError(f"Could not find a value column in FRED response for {series_id}. Columns: {list(frame.columns)}")

    frame[date_col] = pd.to_datetime(frame[date_col])
    frame[value_col] = pd.to_numeric(frame[value_col], errors="coerce")
    return frame[[date_col, value_col]].rename(columns={date_col: "date", value_col: series_id}).set_index("date")


In [ ]:
# 汇总不同宏观序列的覆盖区间和最近观测值。
# Summarize coverage ranges and the latest observations across macro series.
fred_df = pd.concat([load_fred_series(series_id) for series_id in SERIES], axis=1).sort_index()

display(fred_df.tail())

fred_summary = pd.DataFrame(
    {
        "non_null": fred_df.notna().sum(),
        "start": fred_df.apply(lambda s: s.dropna().index.min()),
        "end": fred_df.apply(lambda s: s.dropna().index.max()),
        "latest_value": fred_df.ffill().iloc[-1],
    }
)

display(fred_summary)


In [ ]:
# 把每条序列单独画出来，方便看长期趋势和频率差异。
# Plot each series separately so the long-run trend and frequency differences are easy to see.
fig, axes = plt.subplots(len(SERIES), 1, figsize=(12, 9), sharex=True)

for ax, (series_id, label) in zip(axes, SERIES.items()):
    fred_df[series_id].dropna().plot(ax=ax, linewidth=2)
    ax.set_title(f"{series_id}: {label}")
    ax.set_ylabel(series_id)

plt.tight_layout()
plt.show()


## What these series mean / 这些序列在看什么

- `GDP`：经济总量与增长趋势。  
  `GDP`: overall economic output and growth trend.
- `UNRATE`：就业与失业景气度。  
  `UNRATE`: labor-market condition through unemployment.
- `FEDFUNDS`：货币政策利率水平。  
  `FEDFUNDS`: the policy-rate stance.

这些变量更适合做宏观背景或补充说明，不像 Track A 里的日频市场特征那样直接。  
These variables are better as macro background or supporting context than as direct daily Track A features.
